# 🔄 Notebook 05: Retrain Random Forest + Final Model Evaluation

**Đồ án:** Xây dựng hệ thống E-commerce tích hợp Machine Learning

**Mục tiêu:**
1. Retrain Random Forest **với thêm `segment_id`** từ K-Means (Notebook 03)
2. So sánh hiệu suất RF (không segment) vs RF (có segment)
3. Tổng kết metrics & export model cuối cùng

---

> ⚠️ **Yêu cầu:** Chạy Notebook 01, 02, 03, 04 trước.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
import warnings
import os

from sklearn.model_selection import (
    train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('✅ Import thành công!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/ecommerce-ml'
DATA_DIR = f'{PROJECT_DIR}/data'
MODEL_DIR = f'{PROJECT_DIR}/models'

## 1. Load dữ liệu với Segment Labels

In [ ]:
# Load customer features VỚI segment_id từ Notebook 03
customer_data = pd.read_csv(f'{DATA_DIR}/customer_features_with_segments.csv')

# Load config gốc
with open(f'{DATA_DIR}/config.json', 'r') as f:
    config = json.load(f)

# Load RF model cũ (không có segment) để so sánh
rf_old = joblib.load(f'{MODEL_DIR}/random_forest_model.joblib')

FEATURE_COLUMNS_OLD = config['feature_columns']  # Không có segment_id
TARGET = config['target']

# Feature columns MỚI: thêm segment_id
FEATURE_COLUMNS_NEW = FEATURE_COLUMNS_OLD + ['segment_id']

print(f'✅ Loaded data: {customer_data.shape}')
print(f'\n📋 Features cũ ({len(FEATURE_COLUMNS_OLD)}): {FEATURE_COLUMNS_OLD}')
print(f'\n📋 Features mới ({len(FEATURE_COLUMNS_NEW)}): {FEATURE_COLUMNS_NEW}')
print(f'\n🏷️ Segment distribution:')
print(customer_data['segment_id'].value_counts().sort_index())

## 2. Retrain Random Forest với `segment_id`

In [ ]:
# Chuẩn bị data MỚI (có segment_id)
X_new = customer_data[FEATURE_COLUMNS_NEW].copy()
y = customer_data[TARGET].copy()

X_new = X_new.replace([np.inf, -np.inf], np.nan).fillna(0)

# Train/Test split (cùng random_state để so sánh fair)
X_train_new, X_test_new, y_train, y_test = train_test_split(
    X_new, y, test_size=0.2, random_state=42, stratify=y
)

# Chuẩn bị data CŨ (để so sánh)
X_old = customer_data[FEATURE_COLUMNS_OLD].copy()
X_old = X_old.replace([np.inf, -np.inf], np.nan).fillna(0)
X_train_old, X_test_old, _, _ = train_test_split(
    X_old, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train_new.shape[0]} | Test: {X_test_new.shape[0]}')
print(f'Features cũ: {X_train_old.shape[1]} | Features mới: {X_train_new.shape[1]}')

In [ ]:
# GridSearchCV cho model MỚI
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('🌲 Training Random Forest MỚI (với segment_id)...')
print(f'   GridSearchCV: {np.prod([len(v) for v in param_grid.values()])} × 5 folds')

grid_search_new = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid, cv=cv, scoring='f1', n_jobs=-1, verbose=1
)
grid_search_new.fit(X_train_new, y_train)

rf_new = grid_search_new.best_estimator_

print(f'\n✅ Best params: {grid_search_new.best_params_}')
print(f'   Best CV F1: {grid_search_new.best_score_:.4f}')

## 3. So sánh: RF cũ vs RF mới

In [ ]:
# Predictions RF CŨ (không segment)
y_pred_old = rf_old.predict(X_test_old)
y_prob_old = rf_old.predict_proba(X_test_old)[:, 1]

# Predictions RF MỚI (có segment)
y_pred_new = rf_new.predict(X_test_new)
y_prob_new = rf_new.predict_proba(X_test_new)[:, 1]

# So sánh
models_compare = {
    'RF (Không segment)': (y_pred_old, y_prob_old),
    'RF (Có segment_id) ⭐': (y_pred_new, y_prob_new),
}

comparison = []
for name, (y_pred, y_prob) in models_compare.items():
    comparison.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
    })

comp_df = pd.DataFrame(comparison).set_index('Model')

print('=' * 80)
print('📊 SO SÁNH: Random Forest KHÔNG SEGMENT vs CÓ SEGMENT')
print('=' * 80)
print(comp_df.round(4).to_string())

# Tính improvement
print('\n📈 Improvement:')
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']:
    old_val = comp_df.loc['RF (Không segment)', metric]
    new_val = comp_df.loc['RF (Có segment_id) ⭐', metric]
    diff = new_val - old_val
    arrow = '⬆️' if diff > 0 else ('⬇️' if diff < 0 else '➡️')
    print(f'  {arrow} {metric}: {old_val:.4f} → {new_val:.4f} ({diff:+.4f})')

In [ ]:
# Visualization so sánh
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Metrics comparison
comp_df.T.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='white')
axes[0].set_title('Metrics Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.1)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].legend(fontsize=9)

# 2. ROC Curves
for name, (_, y_prob) in models_compare.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[1].plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title('ROC Curves', fontsize=14, fontweight='bold')
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].legend(fontsize=9)

# 3. Feature Importance (New model)
fi = pd.Series(rf_new.feature_importances_, index=FEATURE_COLUMNS_NEW).sort_values()
colors = ['#f39c12' if f == 'segment_id' else '#3498db' for f in fi.index]
fi.plot(kind='barh', ax=axes[2], color=colors)
axes[2].set_title('Feature Importance (New RF)', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Importance')

plt.suptitle('🏆 Random Forest: Without vs With Segment',
              fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# segment_id importance
seg_imp = rf_new.feature_importances_[FEATURE_COLUMNS_NEW.index('segment_id')]
print(f'\n🏷️ segment_id importance: {seg_imp:.4f} (rank #{len(fi) - list(fi.index).index("segment_id")})')

## 4. Cross-Validation Model Mới

In [ ]:
# 5-Fold CV cho model mới
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring_metrics = ['accuracy', 'f1', 'roc_auc', 'precision', 'recall']

print('🔄 5-Fold Cross-Validation (RF + segment_id):\n')
cv_results_new = {}
for metric in scoring_metrics:
    scores = cross_val_score(rf_new, X_new, y, cv=cv, scoring=metric, n_jobs=-1)
    cv_results_new[metric] = scores
    print(f'  {metric:12s}: {scores.mean():.4f} ± {scores.std():.4f}')

## 5. Confusion Matrix Model Cuối

In [ ]:
# Confusion Matrix so sánh
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (name, (y_pred, _)) in enumerate(models_compare.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues' if idx == 0 else 'Greens',
                ax=axes[idx],
                xticklabels=['Không mua', 'Sẽ mua'],
                yticklabels=['Không mua', 'Sẽ mua'])
    axes[idx].set_title(name, fontsize=13, fontweight='bold')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

plt.suptitle('Confusion Matrix Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Export Final Models

Xuất model RF cuối cùng (có segment_id) để tích hợp vào FastAPI.

In [ ]:
# Scale features mới
scaler_new = StandardScaler()
scaler_new.fit(X_train_new)

# 1. Save RF model cuối cùng (GHI ĐÈ model cũ)
rf_final_path = f'{MODEL_DIR}/random_forest_model.joblib'
joblib.dump(rf_new, rf_final_path)
print(f'✅ RF model (final, with segment) → {rf_final_path}')
print(f'   Size: {os.path.getsize(rf_final_path) / 1024 / 1024:.1f} MB')

# 2. Save updated scaler
scaler_path = f'{MODEL_DIR}/scaler_rfm.joblib'
joblib.dump(scaler_new, scaler_path)
print(f'✅ Scaler (updated) → {scaler_path}')

# 3. Update model metadata
final_metadata = {
    'model_type': 'RandomForestClassifier',
    'version': 'v2_with_segments',
    'best_params': grid_search_new.best_params_,
    'feature_columns': FEATURE_COLUMNS_NEW,
    'target': TARGET,
    'metrics': {
        'accuracy': float(accuracy_score(y_test, y_pred_new)),
        'precision': float(precision_score(y_test, y_pred_new)),
        'recall': float(recall_score(y_test, y_pred_new)),
        'f1_score': float(f1_score(y_test, y_pred_new)),
        'roc_auc': float(roc_auc_score(y_test, y_prob_new)),
    },
    'metrics_without_segment': {
        'accuracy': float(accuracy_score(y_test, y_pred_old)),
        'f1_score': float(f1_score(y_test, y_pred_old)),
        'roc_auc': float(roc_auc_score(y_test, y_prob_old)),
    },
    'cv_f1_mean': float(cv_results_new['f1'].mean()),
    'cv_f1_std': float(cv_results_new['f1'].std()),
    'train_samples': int(len(X_train_new)),
    'test_samples': int(len(X_test_new)),
    'n_features': len(FEATURE_COLUMNS_NEW),
    'segment_id_importance': float(seg_imp),
}

metadata_path = f'{MODEL_DIR}/rf_model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(final_metadata, f, indent=2, default=str)
print(f'✅ Metadata → {metadata_path}')

## 7. Tổng kết tất cả Models

In [ ]:
# Load all configs
with open(f'{MODEL_DIR}/segmentation_config.json', 'r') as f:
    seg_config = json.load(f)

with open(f'{MODEL_DIR}/knn_config.json', 'r') as f:
    knn_config = json.load(f)

print('=' * 80)
print('🏆 TỔNG KẾT: 3 MODELS ĐÃ HOÀN THÀNH')
print('=' * 80)

print(f'''
┌──────────────────────────────────────────────────────────────────┐
│ MODEL 1: Random Forest (Purchase Prediction)           ⭐ MAIN │
├──────────────────────────────────────────────────────────────────┤
│ Features: {len(FEATURE_COLUMNS_NEW)} (RFM + Behavioral + segment_id)               │
│ F1-Score: {final_metadata['metrics']['f1_score']:.4f}                                          │
│ ROC-AUC:  {final_metadata['metrics']['roc_auc']:.4f}                                          │
│ CV F1:    {final_metadata['cv_f1_mean']:.4f} ± {final_metadata['cv_f1_std']:.4f}                                  │
│ Files:    random_forest_model.joblib, scaler_rfm.joblib         │
└──────────────────────────────────────────────────────────────────┘

┌──────────────────────────────────────────────────────────────────┐
│ MODEL 2: K-Means + PCA (Customer Segmentation)                 │
├──────────────────────────────────────────────────────────────────┤
│ Clusters:       {seg_config['n_clusters']}                                               │
│ PCA Components: {seg_config['n_pca_components']}                                               │
│ Silhouette:     {seg_config['silhouette_score']:.4f}                                        │
│ Davies-Bouldin: {seg_config['davies_bouldin_score']:.4f}                                        │
│ Files:          kmeans_model.joblib, pca_transformer.joblib     │
└──────────────────────────────────────────────────────────────────┘

┌──────────────────────────────────────────────────────────────────┐
│ MODEL 3: KNN (Product Recommendation)                          │
├──────────────────────────────────────────────────────────────────┤
│ Products: {knn_config['total_products']:,}                                            │
│ Metric:   Cosine Similarity                                    │
│ Hit Rate: {knn_config['hit_rate']:.4f}                                              │
│ Files:    knn_model.joblib, tfidf_vectorizer.joblib             │
└──────────────────────────────────────────────────────────────────┘

┌──────────────────────────────────────────────────────────────────┐
│ LIÊN KẾT GIỮA CÁC MODELS                                      │
├──────────────────────────────────────────────────────────────────┤
│ K-Means → segment_id → Random Forest feature                   │
│ Random Forest → purchase_probability → KNN prioritization      │
│ Random Forest → threshold → Smart Promotion Popup              │
└──────────────────────────────────────────────────────────────────┘
''')

In [ ]:
# Liệt kê tất cả files đã export
print('📦 FILES ĐÃ EXPORT (sẵn sàng cho FastAPI Backend):')
print('=' * 60)

print(f'\n📁 {MODEL_DIR}/')
total_size = 0
for f in sorted(os.listdir(MODEL_DIR)):
    fpath = os.path.join(MODEL_DIR, f)
    size = os.path.getsize(fpath)
    total_size += size
    if size > 1024 * 1024:
        print(f'  📄 {f:45s} {size/1024/1024:.1f} MB')
    else:
        print(f'  📄 {f:45s} {size/1024:.1f} KB')

print(f'\n📁 {DATA_DIR}/')
for f in sorted(os.listdir(DATA_DIR)):
    fpath = os.path.join(DATA_DIR, f)
    size = os.path.getsize(fpath)
    total_size += size
    if size > 1024 * 1024:
        print(f'  📄 {f:45s} {size/1024/1024:.1f} MB')
    else:
        print(f'  📄 {f:45s} {size/1024:.1f} KB')

print(f'\n💾 Total size: {total_size/1024/1024:.1f} MB')

In [ ]:
# Download tất cả models về local
# Uncomment để download:

# from google.colab import files
# import shutil
#
# # Zip toàn bộ thư mục models
# shutil.make_archive('/content/ecommerce_models', 'zip', MODEL_DIR)
# files.download('/content/ecommerce_models.zip')
#
# # Hoặc zip cả data lẫn models
# shutil.make_archive('/content/ecommerce_ml_full', 'zip', PROJECT_DIR)
# files.download('/content/ecommerce_ml_full.zip')

print('💡 Uncomment cell trên để download files về máy.')
print('   Hoặc download trực tiếp từ Google Drive: ecommerce-ml/models/')
print('\n' + '=' * 60)
print('🎉 SPRINT 2 HOÀN TẤT!')
print('   Tiếp theo: Sprint 3 – FastAPI Backend')
print('=' * 60)